In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from pathlib import Path
from comet_ml import Experiment

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import generate_missing_data
# from imputation import imputation, get_rmse
from experiment_runner import load_dataset, run_single_experiment, log_experiment

np.random.seed(42)

In [2]:
key = Path("../COMET_API_KEY.zshrc").read_text(encoding="utf-8").strip()
os.environ["COMET_API_KEY"] = key

experiment = Experiment(
    api_key=os.environ.get("COMET_API_KEY"),
    project_name="missing-data-imputation",
    workspace=None
)
experiment.set_name("Compare_mean_and_mice")

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/fc602c00aea54094a420ca699d862164



In [9]:
# TODO переделать логирование в comet, чтобы было понятно на чем запускали и что
datasets = ["housing", "adult"]
# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": [{"median_income": "total_rooms"}, 
                {"total_rooms": "population"}, 
                {"housing_median_age": "households"}],
    "adult": [{"occupation": "hours-per-week"},
              {"workclass": "hours-per-week"}, 
              {"hours-per-week": "age"} ],
}
#  глобальный таргет
global_target = {
    "housing": "median_house_value",
    "adult": "income",
}
# ratios = [0.05, 0.20, 0.50]
ratios = [0.05]
mechanisms = ["MCAR", "MAR"]

# добавить обработку для Simple
EXPERIMENT_CONFIG = {
    "Simple": [
        {"num_strategy": "mean"},
        {"num_strategy": "median"},
    ],
    "KNN": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "MICE": [
        {"max_iter": 10},
        {"max_iter": 50},
    ]
}

for dataset_name in datasets:
    df = load_dataset(dataset_name)
    # удаляем глобальный таргет, чтобы случайно не подглядеть
    df = df.drop(columns=[global_target[dataset_name]])
    print(f"Dataset {dataset_name} loaded. Shape: {df.shape}")

    columns_with_missing = [list(d.keys())[0] for d in columns_target[dataset_name]]

    for mechanism in mechanisms:
        for ratio in ratios:

            df_incomplete, mask = generate_missing_data(
                data=df, 
                columns_config=columns_target[dataset_name],
                mechanism=mechanism, 
                ratio=ratio)
            
            print(f"Missing data generated for {dataset_name} | {mechanism} | Ratio: {ratio}")
            # print(f"counts missing: {mask.sum()} / {mask.size}")

            for method, param_list in EXPERIMENT_CONFIG.items():
                for params in param_list:
                    rmse, acc, time_taken = run_single_experiment(
                        df, 
                        df_incomplete, 
                        mask, 
                        method, 
                        params, 
                        columns_with_missing)
                    
                    params_str = "_".join([f"{k}={v}" for k, v in params.items()])

                    # model_accuracy = ... (метрика модели, если есть)

                    # Логируем параметры эксперимента
                    # log_experiment(
                    #     experiment,
                    #     dataset_name,
                    #     mechanism,
                    #     method,
                    #     params_str,
                    #     rmse,
                    #     time_taken
                    # )

                    rmse_str = f"{rmse:.4f}" if rmse is not None else "N/A"
                    acc_str = f"{acc:.4f}" if acc is not None else "N/A"

                    print(f"Done: {dataset_name} | {mechanism} | {method} ({params_str}) | {ratio} | RMSE: {rmse_str} | ACC: {acc_str} | Time: {time_taken:.2f} sec")



Dataset housing loaded. Shape: (20433, 9)
Missing data generated for housing | MCAR | Ratio: 0.05
Done: housing | MCAR | Simple (num_strategy=mean) | 0.05 | RMSE: 0.9999 | ACC: N/A | Time: 0.01 sec
Done: housing | MCAR | Simple (num_strategy=median) | 0.05 | RMSE: 1.0120 | ACC: N/A | Time: 0.02 sec
Done: housing | MCAR | KNN (n_neighbors=3) | 0.05 | RMSE: 0.5930 | ACC: N/A | Time: 2.24 sec
Done: housing | MCAR | KNN (n_neighbors=5) | 0.05 | RMSE: 0.5748 | ACC: N/A | Time: 2.34 sec
Done: housing | MCAR | KNN (n_neighbors=7) | 0.05 | RMSE: 0.5683 | ACC: N/A | Time: 1.78 sec
Done: housing | MCAR | MICE (max_iter=10) | 0.05 | RMSE: 0.6610 | ACC: N/A | Time: 0.17 sec
Done: housing | MCAR | MICE (max_iter=50) | 0.05 | RMSE: 0.6610 | ACC: N/A | Time: 0.17 sec
Missing data generated for housing | MAR | Ratio: 0.05
Done: housing | MAR | Simple (num_strategy=mean) | 0.05 | RMSE: 1.0283 | ACC: N/A | Time: 0.01 sec
Done: housing | MAR | Simple (num_strategy=median) | 0.05 | RMSE: 1.0569 | ACC: N/A

In [ ]:
# experiment.log_table("final_results.csv", df_results)

experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : Compare_mean_and_mice
COMET INFO:     url                   : https://www.comet.com/kam2607kam/missing-data-imputation/fc602c00aea54094a420ca699d862164
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     RMSE_MCAR_MICE_max_iter=10 [3]       : (397.72316275976726, 502.1143857568462)
COMET INFO:     RMSE_MCAR_MICE_max_iter=50 [3]       : (397.72316275976726, 502.1143857568462)
COMET INFO:     RMSE_MCAR_Simple_strategy=mean [3]   : (1182.5674193453801, 1214.7929651071477)
COMET INFO:     RMSE_MCAR_Simple_strategy=median [3] : (1209.2429482650455, 1250.3822342130632)
COMET INFO:     Time_MCAR_MICE_max_iter=10 [3]       : (0.04879170899948804, 0.060